# Sector Dimension Loader

This notebook maintains the `warehouse.dim_sector` dimension table.

**Purpose**: Track industry sectors and hierarchies

**Key Features**:
* Stable surrogate keys for sectors
* Sector hierarchies (industry groups, sub-sectors)
* SCD Type 1 (overwrite on change)

**Source**: `semantic.sem_sector_map`

**Target**: `workspace.warehouse.dim_sector`

In [0]:
# Load sector taxonomy directly from governed metadata
# This ensures warehouse dimension reflects the canonical sector taxonomy

from pyspark.sql import functions as F

METADATA_BASE = "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata"

# Load sector taxonomy
sectors_df = spark.read.csv(
    f"{METADATA_BASE}/sectors.csv",
    header=True,
    inferSchema=True
)

# Load role families to understand sector relationships
role_families_df = spark.read.csv(
    f"{METADATA_BASE}/role_families.csv",
    header=True,
    inferSchema=True
)

# Transform for warehouse dimension
sector_extract_df = sectors_df.select(
    F.col("sector_key"),
    F.col("sector_name"),
    F.col("parent_sector"),
    F.col("naics_code"),
    F.when(F.col("parent_sector").isNull(), "PRIMARY").otherwise("SECONDARY").alias("sector_category"),
    F.when(F.col("parent_sector").isNull(), 1).otherwise(2).alias("sector_level"),
    F.lit(True).alias("is_active"),
    F.split(F.col("keywords"), "\\|").alias("keywords")
)

sector_extract_df.createOrReplaceTempView("sector_extract")

print(f"Loaded {sector_extract_df.count()} sectors from metadata taxonomy")
display(sector_extract_df)

In [0]:
%sql
-- Generate or maintain stable surrogate keys
CREATE OR REPLACE TEMP VIEW sector_with_keys AS
SELECT
  COALESCE(d.sector_sk, ROW_NUMBER() OVER (ORDER BY s.sector_key) + COALESCE(max_sk, 0)) as sector_sk,
  s.sector_key,
  s.sector_name,
  s.sector_category,
  s.sector_level,
  s.parent_sector,
  s.naics_code,
  s.keywords,
  COALESCE(s.is_active, TRUE) as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM sector_extract s
LEFT JOIN workspace.warehouse.dim_sector d
  ON s.sector_key = d.sector_key
CROSS JOIN (
  SELECT COALESCE(MAX(sector_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_sector
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
-- First, ensure table exists with updated schema
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_sector (
  sector_sk BIGINT,
  sector_key STRING,
  sector_name STRING,
  sector_category STRING,
  sector_level INT,
  parent_sector STRING,
  naics_code STRING,
  keywords ARRAY<STRING>,
  is_active BOOLEAN,
  updated_at TIMESTAMP
) USING DELTA;

MERGE INTO workspace.warehouse.dim_sector target
USING sector_with_keys source
ON target.sector_key = source.sector_key
WHEN MATCHED THEN UPDATE SET
  target.sector_name = source.sector_name,
  target.sector_category = source.sector_category,
  target.sector_level = source.sector_level,
  target.parent_sector = source.parent_sector,
  target.naics_code = source.naics_code,
  target.keywords = source.keywords,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  sector_sk,
  sector_key,
  sector_name,
  sector_category,
  sector_level,
  parent_sector,
  naics_code,
  keywords,
  is_active,
  updated_at
) VALUES (
  source.sector_sk,
  source.sector_key,
  source.sector_name,
  source.sector_category,
  source.sector_level,
  source.parent_sector,
  source.naics_code,
  source.keywords,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate sector dimension and hierarchy
SELECT 
  COUNT(*) as total_sectors,
  COUNT(DISTINCT sector_category) as categories,
  COUNT(DISTINCT parent_sector) as parent_sectors,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_sectors
FROM workspace.warehouse.dim_sector;

-- Show sector hierarchy
SELECT 
  sector_sk,
  sector_name,
  sector_category,
  sector_level,
  parent_sector,
  is_active
FROM workspace.warehouse.dim_sector
ORDER BY sector_category, sector_name
LIMIT 20;